Step 1: Install and Import Packages

In [23]:
!pip install gensim

In [24]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
import plotly.express as px

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Step 2: Load the Base Movie Dataset


In [47]:
df=pd.read_csv("/content/TMDB_movie_dataset_v11.csv",encoding="latin1", on_bad_lines='skip')

In [26]:
df

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,en,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili"
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,en,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,en,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin"
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish"
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,en,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994909,660911,The Perfect 10 Boys,0.000,0,Released,2002-01-01,0,235,True,NaN,...,en,The Perfect 10 Boys,Hot compilation ending in cum-eating! First Ro...,0.600,/pxSF5AajyoV68BTnZ9RttesAjRC.jpg,NaN,NaN,Manzworld Productions,NaN,English
994910,660914,A Mind's Eye,0.000,0,Released,2009-03-26,0,13,False,NaN,...,en,A Mind's Eye,This is a film about Plato's ideas. It is abou...,0.602,NaN,NaN,Drama,Heraclitus Pictures,NaN,English
994911,660915,Flawless Tits 2,0.000,0,Released,2018-09-19,0,156,True,/zBhgmm1kbtssMRKQPl3Qu8cwiaM.jpg,...,en,Flawless Tits 2,It's how she looks when she's laying flat on h...,0.633,/laf0vVaWmQfyZnfyZwAp27FhQuF.jpg,NaN,NaN,BANG!,United States of America,English
994912,660918,The Circle,0.000,0,Released,1972-12-12,0,95,False,/gp94vlxBhU94XqlnqlNuUCCgM4l.jpg,...,en,The Circle,"Two young lovers, Mary and Tim, deal with pres...",0.600,/u47GzoihhbGKVVFEwbaqPXIaMcp.jpg,The Circle is LOVE. The Circle is CONFLICT. Th...,Romance,NaN,United States of America,NaN


In [27]:
df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages'],
      dtype='object')

In [28]:
for i in df.columns:
  print(df[i].iloc[0])

27205
Inception
8.364
34495
Released
2010-07-15
825532764
148
False
/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg
160000000
https://www.warnerbros.com/movies/inception
tt1375666
en
Inception
Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious.
83.952
/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg
Your mind is the scene of the crime.
Action, Science Fiction, Adventure
Legendary Pictures, Syncopy, Warner Bros. Pictures
United Kingdom, United States of America
English, French, Japanese, Swahili


Step 3: Handle Missing Values and Combine Text Features


We don't need any data which do not have imdb_id because that's the identifier for everything we are going to do.

In [62]:
df=df.dropna(subset=['imdb_id'])

In [63]:
df

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,combined_text,extracted_features,tokens
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure",Legendary Pictures,"United Kingdom, United States of America","English, French, Japanese, Swahili","Your mind is the scene of the crime. Cobb, a s...",Title Inception Production CompaniesLegendary ...,"[title, inception, production, companieslegend..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction",Legendary Pictures,"United Kingdom, United States of America",English,Mankind was born on Earth. It was never meant ...,Title Interstellar Production CompaniesLegenda...,"[title, interstellar, production, companiesleg..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller",DC Comics,"United Kingdom, United States of America","English, Mandarin",Welcome to a world without rules. Batman raise...,Title The Dark Knight Production CompaniesDC C...,"[title, dark, knight, production, companiesdc,..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction",Dune Entertainment,"United States of America, United Kingdom","English, Spanish",Enter the world of Pandora. In the 22nd centur...,Title Avatar Production CompaniesDune Entertai...,"[title, avatar, production, companiesdune, ent..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian",Some assembly required. When an unexpected ene...,Title The Avengers Production CompaniesMarvel ...,"[title, avenger, production, companiesmarvel, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994904,660907,First Times & Second Chances,0.000,0,Released,2019-05-27,0,106,True,/nYSvs63OzYjjsvVyB5VFwU32URR.jpg,...,0.601,/hstKsbDN2jsYWhOlRyiorjpP2xT.jpg,,Romance,Bellesa Films,United States of America,English,While spending some alone time with her frien...,Title First Times & Second Chances Production ...,"[title, first, time, second, chance, productio..."
994906,660908,Kai susilieja upÄs,0.000,0,Released,1961-05-05,0,82,False,NaN,...,0.600,/yPuKqQvDrQfEbMRKtwqRMLPUaLM.jpg,,Drama,Lithuanian Film Studio,Soviet Union,Lithuanian,,Title Kai susilieja upÄs Production Companies...,"[title, kai, susilieja, ups, production, compa..."
994907,660909,Bajo otro sol,0.000,0,Released,1988-08-11,0,0,False,/4vngSvh2EtM5HR7gC7tPqKo3gRy.jpg,...,0.600,/1CU9MOaG5acgOtrVFJyktlEmEnB.jpg,,Drama,None,NaN,Spanish,Student/radical returns to Argentina after ye...,Title Bajo otro sol Production CompaniesNone R...,"[title, bajo, otro, sol, production, companies..."
994910,660914,A Mind's Eye,0.000,0,Released,2009-03-26,0,13,False,NaN,...,0.602,NaN,,Drama,Heraclitus Pictures,NaN,English,This is a film about Plato's ideas. It is abo...,Title A Mind's Eye Production CompaniesHeracli...,"[title, mind, eye, production, companiesheracl..."


Fill the null with 'None' missing indicator

In [66]:
df.loc[:,'production_companies'] = df['production_companies'].fillna('None')
df.loc[:,'production_companies'] = df['production_companies'].apply(lambda x: x.split(',')[0].strip() if x != 'None' else 'None')

Extract all the Genres, if null impute 'None'

In [67]:
df.loc[:,'genres'] = df['genres'].fillna('None')

Combine tagline + overview for better context of movie

In [68]:
df.loc[:,'tagline'] = df['tagline'].fillna('')
df.loc[:,'overview'] = df['overview'].fillna('')
df.loc[:,'combined_text'] = df['tagline'] + ' ' + df['overview']

Finally combine all extracted features into a single column

In [69]:
df.loc[:,'extracted_features'] = ('Title '+
    df['title'].astype(str) + ' Production Companies' +
    df['production_companies'].astype(str) + ' Rating ' +
    df['vote_average'].astype(str) +  ' Language ' +
    df['original_language'].astype(str) + ' Genres ' +
    df['genres'].astype(str) + ' context ' +
    df['combined_text'].astype(str)
)

Step 4: Preprocess Text and Tokenize for NLP


In [71]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]
    return cleaned_tokens

df.loc[:,'tokens'] = df['extracted_features'].apply(clean_and_tokenize)

Step 5: Train Word2Vec Model


In [72]:
tokenized_corpus = df['tokens'].tolist()

w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=30,
    window=5,
    min_count=1,
    workers=4
)

Step 6: Create Document Vectors by Averaging Word Vectors


In [73]:
def get_document_vector(tokens, model, vector_size=30):
    valid_words = [word for word in tokens if word in model.wv]
    if not valid_words:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(model.wv[valid_words], axis=0)

vector_size = 30
document_vectors = [
    get_document_vector(tokens, w2v_model, vector_size)
    for tokens in df['tokens']
]

Step 7: Export to text_vectors.csv


In [79]:
# Convert vectors to float16 to reduce memory/precision
text_vectors = np.array([get_document_vector(tokens, w2v_model) for tokens in df['tokens']], dtype=np.float16)

df_vectors = pd.DataFrame({
    'imdb_id': df['imdb_id'],
    'rounded_vectors': [str(list(np.round(vec, 2))) for vec in text_vectors]
})

df_vectors.to_csv("text_vectors.csv", index=True)
df_vectors.head()

,imdb_id,rounded_vectors
0,tt1375666,"[np.float16(0.94), np.float16(-2.6), np.float1..."
1,tt0816692,"[np.float16(0.55), np.float16(-2.64), np.float..."
2,tt0468569,"[np.float16(1.08), np.float16(-2.35), np.float..."
3,tt0499549,"[np.float16(0.97), np.float16(-3.09), np.float..."
4,tt0848228,"[np.float16(0.97), np.float16(-2.57), np.float..."


Step 8: Validate Vectors Using PCA and 3D Plotly Visualization


In [75]:
from sklearn.decomposition import PCA

matrix_representation = np.vstack(document_vectors)

pca = PCA(n_components=3)
reduced_vectors = pca.fit_transform(matrix_representation)

df_pca = pd.DataFrame(reduced_vectors, columns=['PCA1', 'PCA2','PCA3'])
df_pca['title'] = df['title']
df_pca['genres'] = df['genres']

fig = px.scatter_3d(
    df_pca.head(200),
    x='PCA1',
    y='PCA2',
    z='PCA3',

    hover_data=['genres','title'],
    title="3D PCA Visualization of Movie Word2Vec Vectors"
)

fig.update_traces(marker=dict(size=4, opacity=0.8))
fig.show()

In [76]:
df_vectors.shape

(569928, 2)